# Swarm-style helicorder (Bokeh)

This notebook demonstrates `SwarmHelicorderBk` for strip-style dayplots with interactive focus decoding and optional clipboard linking.

**Current Bokeh parity deltas vs MPL:**
- Timezone footer labels are rendered as y-tick/footer text (not mirrored MPL left/right axis styling).
- Clipboard sync uses a centered focus window policy around `focus_time`.
- Catalog pick matching prefers full SEED id and falls back to station matching when needed.
- In standalone HTML exports, Python event callbacks are not active (focus sync is interactive in notebook/server contexts).


In [ ]:
from pathlib import Path

from obspy import UTCDateTime, read
from obspy.core.event import Catalog, Event, Origin, Pick, WaveformStreamID

from vdapseisutils.core.swarmmpl.bokeh import SwarmHelicorderBk

REPO = Path.cwd().resolve().parent if Path.cwd().name == "gallery" else Path.cwd().resolve()


### Example 1: Basic dayplot strips

Create a simple Bokeh helicorder and display it inline.


In [ ]:
path = REPO / "data/waveforms/gareloi_test_data_20220710-010000.mseed"
st = read(path)
st.trim(UTCDateTime("2022-07-10T01:00:00"), UTCDateTime("2022-07-10T02:00:00"))

heli = SwarmHelicorderBk(st, interval=60, color="swarm", title="Gareloi Bokeh helicorder")
heli.show()


### Example 2: Interval + styling + tick labels

Customize strip interval, color scheme, and timezone/tick labeling.


In [ ]:
path = REPO / "data/waveforms/gareloi_test_data_20220710-010000.mseed"
st = read(path)
st.trim(UTCDateTime("2022-07-10T01:00:00"), UTCDateTime("2022-07-10T03:00:00"))

heli2 = SwarmHelicorderBk(
    st,
    interval=30,
    color="earthworm",
    title="Bokeh helicorder: 30 min strips",
    utc_offset_left="UTC",
    utc_offset_right="UTC+00",
)
heli2.set_tticks(label_spacing=2)
heli2.set_tzticklabel(custom="UTC+00", axes="left")
heli2.show()


### Example 3: Tags/catalog markers + HTML export

Add tag/catalog markers, optionally attach a clipboard view, and write standalone HTML.


In [ ]:
path = REPO / "data/waveforms/Augustine_test_data_FI.mseed"
st = read(path)
st.sort()
st.trim(st[0].stats.starttime, st[0].stats.endtime)

heli3 = SwarmHelicorderBk(st, interval=15, color="obspy", title="Augustine markers")
t0 = st[0].stats.starttime + 2
t1 = st[0].stats.starttime + 5
heli3.plot_tags([t0, t1], marker="diamond", color="red", markersize=12)

event = Event(
    origins=[Origin(time=t0 + 30)],
    picks=[
        Pick(
            time=t0 + 45,
            phase_hint="P",
            waveform_id=WaveformStreamID(
                network_code=st[0].stats.network,
                station_code=st[0].stats.station,
                location_code=st[0].stats.location,
                channel_code=st[0].stats.channel,
            ),
        )
    ],
)
heli3.plot_catalog(Catalog(events=[event]), plot_picks=True, plot_origins=True)

# Optional: attach a linked clipboard view. Focus-sync window is centered on focus_time.
heli3.attach_clipboard(mode="wg", window_s=600, sync_focus=True)

out = REPO / "gallery/SwarmMPL/helicorder_tutorial_bokeh_example1.html"
heli3.save(out, title="SwarmHelicorderBk Example")
out
